
# Toy TFT with real outage data and fake weather

This notebook is a stepping stone while the Earth Engine export runs.

It:
- loads the real POUS outage time series,
- standardizes the columns if needed,
- creates synthetic weather covariates with the same names you will later get from GEE,
- fits a very small TFT on a subset,
- prints a baseline and a model result,
- shows a quick prediction plot.

The weather is fake. The outage target is real.


In [1]:

# If needed, install dependencies first in your environment:
# pip install lightning pytorch-forecasting torch pandas numpy matplotlib pyarrow

try:
    import lightning.pytorch as pl
    print('Using lightning.pytorch')
except ImportError:
    import pytorch_lightning as pl
    print('Using pytorch_lightning')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE

pl.seed_everything(42)


Using lightning.pytorch


Seed set to 42


42

In [2]:

# Paths on your machine.
POUS_CSV = r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\POUS.csv"
TIMESERIES_PQ = r"C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\timeseries.pq"

# Model settings.
MAX_ENCODER_LENGTH = 72
MAX_PREDICTION_LENGTH = 12
BATCH_SIZE = 64
SEED = 42


In [3]:

def load_table(path):
    if path.lower().endswith('.pq') or path.lower().endswith('.parquet'):
        return pd.read_parquet(path)
    return pd.read_csv(path)


def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(x) for x in tup if str(x) != '']) for tup in df.columns]
    return df


def rename_first_match(df, target, candidates):
    for c in candidates:
        if c in df.columns:
            if c != target:
                df = df.rename(columns={c: target})
            return df, c
    return df, None


def standardize_timeseries(df):
    df = flatten_columns(df.copy())
    df = df.reset_index()
    df.columns = [str(c) for c in df.columns]

    rename_map = {
        'county': ['county', 'County', 'CountyFIPS', 'county_id', 'CountyID', 'FIPS', 'geoid', 'GEOID'],
        'datetime': ['datetime', 'DateTime', 'date_time', 'timestamp', 'time', 'Date'],
        'outageFraction': ['outageFraction', 'OutageFraction', 'outage_fraction', 'POUS', 'pous'],
        'customersTracked': ['customersTracked', 'CustomersTracked', 'customers_tracked', 'TrackedCustomers'],
    }

    found = {}
    for target, candidates in rename_map.items():
        df, src = rename_first_match(df, target, candidates)
        found[target] = src

    # If county is still missing, try the first column that looks like an identifier.
    if 'county' not in df.columns:
        raise ValueError(f"Could not find a county/FIPS column. Available columns: {list(df.columns)}")
    if 'datetime' not in df.columns:
        raise ValueError(f"Could not find a datetime column. Available columns: {list(df.columns)}")
    if 'outageFraction' not in df.columns:
        raise ValueError(f"Could not find an outageFraction column. Available columns: {list(df.columns)}")

    df['county'] = df['county'].astype(str).str.zfill(5)
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df = df.dropna(subset=['datetime', 'county', 'outageFraction']).copy()
    df = df.sort_values(['county', 'datetime']).reset_index(drop=True)

    # Add a time index that is integer hours from the first timestamp in each county.
    df['time_idx'] = (
        (df['datetime'] - df.groupby('county')['datetime'].transform('min'))
        .dt.total_seconds() // 3600
    ).astype(int)

    # Calendar features.
    df['hour'] = df['datetime'].dt.hour.astype(int)
    df['dayofweek'] = df['datetime'].dt.dayofweek.astype(int)
    df['month'] = df['datetime'].dt.month.astype(int)

    return df


In [5]:

# Load the real outage data.
raw_timeseries = load_table(TIMESERIES_PQ)
raw_pous = load_table(POUS_CSV)

print('timeseries.pq shape:', raw_timeseries.shape)
print('timeseries.pq columns:', list(raw_timeseries.columns)[:30])
print('POUS.csv shape:', raw_pous.shape)
print('POUS.csv columns:', list(raw_pous.columns)[:30])

print('Head of timeseries.pq:')
display(raw_timeseries.head())


timeseries.pq shape: (159522635, 2)
timeseries.pq columns: ['OutageFraction', 'CustomersTracked']
POUS.csv shape: (1040, 13)
POUS.csv columns: ['event_start', 'CountyFIPS', 'county_pop', 'pre_outage_tracked_customers', 'days_since_data_start', 'duration_hours', 'n_periods', 'integral', 'pop_hours_supply_lost', 'storm', 'duration_days', 'year', 'month']
Head of timeseries.pq:


OutageFraction  CustomersTracked
RecordDateTime CountyFIPS                                  
2017-01-01     01001                  0.0               0.0
               01003                  0.0               0.0
               01005                  0.0               0.0
               01007                  0.0               0.0
               01009                  0.0               0.0

In [7]:
print(raw_timeseries.shape)
print(type(raw_timeseries.index))
print(raw_timeseries.index.names)
print(raw_timeseries.head())

(159522635, 2)
<class 'pandas.core.indexes.multi.MultiIndex'>
['RecordDateTime', 'CountyFIPS']
                           OutageFraction  CustomersTracked
RecordDateTime CountyFIPS                                  
2017-01-01     01001                  0.0               0.0
               01003                  0.0               0.0
               01005                  0.0               0.0
               01007                  0.0               0.0
               01009                  0.0               0.0


^ this is huge (160 million rows). perhaps instead of the full time series, only include windows 24hrs before an event to 24hrs after?
- or if don't know the 'end' of the event a priori, take 10 days. But i think we should know the duration for each, given the POUS.csv


In [6]:

# Standardize the time series table.
df = standardize_timeseries(raw_timeseries)

print('Standardized columns:', list(df.columns))
print('Rows:', len(df))
print('Counties:', df['county'].nunique())
print('Datetime range:', df['datetime'].min(), 'to', df['datetime'].max())
print(df[['county', 'datetime', 'outageFraction']].head())

# Keep a smaller toy subset so the model runs quickly.
county_order = (
    df.groupby('county')['outageFraction']
      .sum()
      .sort_values(ascending=False)
      .head(6)
      .index
)
df = df[df['county'].isin(county_order)].copy()
print('Toy counties:', list(county_order))


MemoryError: Unable to allocate 1.19 GiB for an array with shape (159522635,) and data type uint64

In [ ]:

# Create fake weather with the same columns as the future GEE export.
# These are synthetic. The point is to get the TFT plumbing working.

rng = np.random.default_rng(SEED)

# County-specific offsets so each county gets slightly different weather.
county_code = df['county'].astype('category').cat.codes.to_numpy()

# A smooth storminess signal with county variation and time variation.
storm_phase = (
    0.04 * df['time_idx'].to_numpy()
    + 0.9 * county_code
    + 0.2 * np.sin(df['hour'].to_numpy() / 24 * 2 * np.pi)
)
storminess = 1 / (1 + np.exp(-(np.sin(storm_phase) + 0.25 * rng.normal(size=len(df)))))

# Fake weather columns.
df['gust_mps'] = np.clip(4 + 28 * storminess + rng.normal(0, 1.2, len(df)), 0, None)
df['wind_speed_mps'] = np.clip(0.55 * df['gust_mps'] + rng.normal(0, 0.8, len(df)), 0, None)
df['precip_mm'] = np.clip(0.12 * np.maximum(df['gust_mps'] - 8, 0) + rng.gamma(1.2, 0.25, len(df)), 0, None)
df['pressure_hpa'] = 1014 - 0.9 * np.maximum(df['gust_mps'] - 10, 0) + rng.normal(0, 0.7, len(df))
df['temp_c'] = 27 - 0.08 * np.maximum(df['gust_mps'] - 8, 0) + 4 * np.sin(2 * np.pi * df['month'] / 12) + rng.normal(0, 1.0, len(df))

print(df[['county', 'datetime', 'outageFraction', 'gust_mps', 'wind_speed_mps', 'precip_mm', 'pressure_hpa', 'temp_c']].head())


In [ ]:

# Visual sanity check.
fig, ax1 = plt.subplots(figsize=(11, 4))
for county, g in df.groupby('county'):
    g = g.sort_values('time_idx')
    ax1.plot(g['time_idx'], g['outageFraction'], linewidth=1, alpha=0.8, label=f'outage {county}')
ax1.set_xlabel('time_idx')
ax1.set_ylabel('outageFraction')
ax1.set_title('Real outage data in the toy subset')
plt.show()

fig, ax2 = plt.subplots(figsize=(11, 4))
first_county = df['county'].iloc[0]
g = df[df['county'] == first_county].sort_values('time_idx')
ax2.plot(g['time_idx'], g['outageFraction'], label='outageFraction')
ax2.plot(g['time_idx'], g['gust_mps'], label='gust_mps')
ax2.set_xlabel('time_idx')
ax2.set_title(f'Example county {first_county}: outage vs fake gust')
ax2.legend()
plt.show()


In [ ]:

# Train/validation split.
training_cutoff = df['time_idx'].max() - MAX_PREDICTION_LENGTH

training = TimeSeriesDataSet(
    df[df.time_idx <= training_cutoff],
    time_idx='time_idx',
    target='outageFraction',
    group_ids=['county'],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=['county'],
    time_varying_known_reals=['time_idx', 'hour', 'dayofweek', 'month'],
    time_varying_unknown_reals=['outageFraction', 'gust_mps', 'wind_speed_mps', 'precip_mm', 'pressure_hpa', 'temp_c'],
    target_normalizer=GroupNormalizer(groups=['county']),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df,
    predict=True,
    stop_randomization=True,
)

train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=0)

print('Training samples:', len(training))
print('Validation samples:', len(validation))
print('Train batches:', len(train_dataloader))
print('Val batches:', len(val_dataloader))


In [ ]:

# Baseline.
baseline = Baseline()
baseline_pred = baseline.predict(val_dataloader, return_y=True)
baseline_mae = MAE()(baseline_pred.output, baseline_pred.y)
print('Baseline MAE:', float(baseline_mae))


In [ ]:

# Tiny TFT.
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=8,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=3,
)

print('Model parameters:', tft.size())

try:
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        LearningRateMonitor(),
    ]
except Exception:
    callbacks = []

trainer = pl.Trainer(
    max_epochs=5,
    accelerator='auto',
    devices='auto',
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    logger=True,
    callbacks=callbacks,
)

trainer.fit(tft, train_dataloader, val_dataloader)


In [ ]:

# Validation and plot.
pred = tft.predict(val_dataloader, return_y=True)
tft_mae = MAE()(pred.output, pred.y)
print('TFT MAE:', float(tft_mae))

raw_predictions, x = tft.predict(val_dataloader, mode='raw', return_x=True)
fig = tft.plot_prediction(x, raw_predictions, idx=0, add_loss_to_title=True)
plt.show()
